# Day3_14 · 여러 분할에서 모델 비교

같은 콘크리트 자료를 여러 번 나누어 선형회귀·Ridge·랜덤포레스트의 점수와 흔들림을 비교합니다.

## 이 노트북에서 확인할 내용

- 다섯 부분 교차검증의 평가 순서를 설명할 수 있습니다.
- 모형별 평균 점수와 분할 사이 변동을 비교할 수 있습니다.
- 일반 표 자료와 시간자료의 분할 방법을 구분할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-14-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-14-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-14-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-14-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-14-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-14-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-14-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-14-08 · 입력 열 정하기

In [ ]:
feature_columns = ['cement_kg_m3', 'slag_kg_m3', 'fly_ash_kg_m3', 'water_kg_m3', 'superplasticizer_kg_m3', 'coarse_aggregate_kg_m3', 'fine_aggregate_kg_m3', 'age_days']

In [ ]:
pd.Series(feature_columns)

### D3B-14-09 · 입력표 만들기

In [ ]:
X = concrete[feature_columns]

In [ ]:
X.head()

### D3B-14-10 · 결과 열 만들기

In [ ]:
y = concrete['strength_mpa']

In [ ]:
y.head()

## 한 번의 분할 대신 여러 번 평가하기

자료를 다섯 부분으로 나누고, 네 부분으로 학습한 뒤 남은 한 부분을 평가합니다. 평가 부분을 바꾸어 다섯 번 반복하면 분할에 따른 점수 변화를 볼 수 있습니다.

### D3B-14-11 · 교차검증 도구 불러오기

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

### D3B-14-12 · 다섯 부분을 나누는 규칙 만들기

In [ ]:
five_folds = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
five_folds

### D3B-14-13 · 비교할 회귀모형 불러오기

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

### D3B-14-14 · 선형회귀의 다섯 RMSE 계산

In [ ]:
linear_rmse = -cross_val_score(LinearRegression(), X, y, cv=five_folds, scoring='neg_root_mean_squared_error')

In [ ]:
pd.Series(linear_rmse, name='Linear Regression RMSE').round(2)

### D3B-14-15 · Ridge 회귀의 다섯 RMSE 계산

In [ ]:
ridge_rmse = -cross_val_score(Ridge(alpha=10.0), X, y, cv=five_folds, scoring='neg_root_mean_squared_error')

In [ ]:
pd.Series(ridge_rmse, name='Ridge RMSE').round(2)

### D3B-14-16 · 랜덤포레스트의 다섯 RMSE 계산

In [ ]:
forest_rmse = -cross_val_score(RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=42, n_jobs=-1), X, y, cv=five_folds, scoring='neg_root_mean_squared_error')

In [ ]:
pd.Series(forest_rmse, name='Random Forest RMSE').round(2)

### D3B-14-17 · 모든 분할 결과를 한 표로 만들기

In [ ]:
fold_scores = pd.DataFrame({
    'Linear Regression': linear_rmse,
    'Ridge': ridge_rmse,
    'Random Forest': forest_rmse,
})

In [ ]:
fold_scores.round(2)

### D3B-14-18 · 모형별 평균과 표준편차 계산

In [ ]:
score_summary = fold_scores.agg(['mean', 'std']).T

In [ ]:
score_summary.round(2)

### D3B-14-19 · 분할별 RMSE 분포 그리기

In [ ]:
score_long = fold_scores.melt(var_name='model', value_name='RMSE')
sns.boxplot(data=score_long, x='model', y='RMSE')
sns.stripplot(data=score_long, x='model', y='RMSE', color='black', size=6)
plt.xticks(rotation=15)
plt.show()

평균 RMSE가 작을수록 예측오차가 작습니다. 표준편차와 점의 범위는 자료 분할이 바뀔 때 점수가 얼마나 흔들렸는지 보여 줍니다.

### D3B-14-20 · 평균 RMSE가 가장 작은 모형 찾기

In [ ]:
selected_model_name = score_summary['mean'].idxmin()

In [ ]:
selected_model_name

### D3B-14-21 · 선택 근거 표 만들기

In [ ]:
selection_table = score_summary.rename(columns={'mean': 'mean_RMSE', 'std': 'RMSE_std'}).sort_values('mean_RMSE')

In [ ]:
selection_table.round(2)

Ridge는 입력변수 계수가 지나치게 커지는 것을 제한하는 선형모형입니다. 랜덤포레스트는 비선형 관계를 표현할 수 있지만, 어떤 자료에서든 항상 더 좋은 것은 아닙니다.

### D3B-14-22 · 바꾸어 볼 Ridge 제한값 정하기

In [ ]:
# 0.1, 1, 10, 100 중 하나로 바꾸어 보세요.
ridge_alpha = 1.0

In [ ]:
ridge_alpha

### D3B-14-23 · 선택한 제한값의 다섯 RMSE 계산

In [ ]:
chosen_ridge_rmse = -cross_val_score(Ridge(alpha=ridge_alpha), X, y, cv=five_folds, scoring='neg_root_mean_squared_error')

In [ ]:
pd.Series(chosen_ridge_rmse, name=f'Ridge alpha={ridge_alpha}').round(2)

### D3B-14-24 · Ridge 제한값 결과 요약

In [ ]:
chosen_ridge_summary = pd.Series({'mean_RMSE': chosen_ridge_rmse.mean(), 'RMSE_std': chosen_ridge_rmse.std(ddof=1)})

In [ ]:
chosen_ridge_summary.round(2)

이 다섯 부분 교차검증은 서로 독립적인 배합 행을 가정한 예입니다. 날짜 순서가 있는 기상자료에는 행을 무작위로 섞지 않고 이전에 배운 시간순 검증을 사용합니다.

### D3B-14-25 · 모형 비교 결론 틀 만들기

In [ ]:
validation_note = pd.Series({
    '선택한 모형': selected_model_name,
    '비교 기준': '동일한 다섯 분할의 RMSE',
    '함께 기록': '평균 RMSE와 분할 사이 표준편차',
    '적용하지 않을 자료': '시간 순서가 중요한 기상 시계열',
})

In [ ]:
validation_note

### D3B-14-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''X, y, five_folds를 그대로 사용하세요. 기존 셀은 수정하지 마세요. Linear Regression, Ridge, Random Forest의 MAE를 같은 다섯 분할에서 비교하는 새 셀을 제안해 주세요. 기존 RMSE 표와 새 MAE 표에서 모형 순위가 같은지 확인하는 표도 만들어 주세요.'''.strip()

In [ ]:
codex_request

### D3B-14-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['세 모형이 동일한 분할을 사용하는가?', 'RMSE와 MAE를 같은 열에 섞지 않는가?', '시간자료에 무작위 K-fold를 권하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist